In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score
import numpy as np
import os
import seaborn as sns

sns.set_context("notebook")

# --- Clinical palette ---
CLINICAL_COLORS = {
    "ViennaAIdb": "#005B96",
    "MIMIC": "#00A6A6",
    "Reference": "black"
}


def calculate_auprc_ci(y_true, y_pred, n_bootstraps=20, rng_seed=42):
    """
    Calculates the 95% CI for AUPRC using bootstrapping.
    Returns: (auprc, lower, upper, prec_lower, prec_upper, mean_recall)
    """
    rng = np.random.RandomState(rng_seed)
    bootstrapped_auprc = []
    precisions = []
    base_recall = np.linspace(0, 1, 101)

    original_auprc = average_precision_score(y_true, y_pred)

    print(f"Bootstrapping CI for AUPRC ({n_bootstraps} iterations)...")

    for i in range(n_bootstraps):
        indices = rng.randint(0, len(y_pred), len(y_pred))

        if len(np.unique(y_true[indices])) < 2:
            continue

        unique_idx = np.unique(indices)
        prec_boot, rec_boot, _ = precision_recall_curve(y_true[unique_idx], y_pred[unique_idx])
        score = average_precision_score(y_true[unique_idx], y_pred[unique_idx])
        bootstrapped_auprc.append(score)

        # PR curves go from high recall to low, so flip for interpolation
        sorted_order = np.argsort(rec_boot)
        rec_sorted = rec_boot[sorted_order]
        prec_sorted = prec_boot[sorted_order]

        prec_interp = np.interp(base_recall, rec_sorted, prec_sorted)
        precisions.append(prec_interp)

    sorted_scores = np.array(bootstrapped_auprc)
    sorted_scores.sort()

    lower = sorted_scores[int(0.025 * len(sorted_scores))]
    upper = sorted_scores[int(0.975 * len(sorted_scores))]

    precisions = np.array(precisions)
    prec_lower = np.percentile(precisions, 2.5, axis=0)
    prec_upper = np.percentile(precisions, 97.5, axis=0)

    return original_auprc, lower, upper, prec_lower, prec_upper, base_recall


def plot_precision_recall():
    # --- Configuration ---
    output_dir = 'model_outputs'
    muw_file = os.path.join(output_dir, 'muw_results.npz')
    mimic_file = os.path.join(output_dir, 'mimic_results.npz')
    save_path = 'PRC_with_CI.png'

    # --- Load Data ---
    data_loaded = {}
    for name, filepath in [('Internal', muw_file), ('External', mimic_file)]:
        if os.path.exists(filepath):
            data = np.load(filepath)
            data_loaded[name] = {
                'y_true': data['y_true'],
                'y_pred': data['y_pred_proba'],
            }
            print(f"Loaded {name} data from {filepath}")
        else:
            print(f"Warning: {filepath} not found.")

    if not data_loaded:
        print("No data found.")
        return

    sns.set_theme(style="whitegrid", palette="pastel")
    fig, ax = plt.subplots()

    prevalences = {}

    # --- Internal ---
    if 'Internal' in data_loaded:
        d = data_loaded['Internal']
        prevalence = np.mean(d['y_true'])
        prevalences['ViennaAIdb'] = prevalence

        auprc, low, high, prec_low, prec_high, mean_recall = calculate_auprc_ci(d['y_true'], d['y_pred'])
        precision, recall, _ = precision_recall_curve(d['y_true'], d['y_pred'])

        ax.plot(recall, precision,
                color=CLINICAL_COLORS['ViennaAIdb'], lw=1,
                label=f'ViennaAIdb (AUPRC: {auprc:.2f} [{low:.2f}-{high:.2f}])')
        ax.fill_between(mean_recall, prec_low, prec_high,
                        color=CLINICAL_COLORS['ViennaAIdb'], alpha=0.2)

    # --- External ---
    if 'External' in data_loaded:
        d = data_loaded['External']
        prevalence = np.mean(d['y_true'])
        prevalences['MIMIC'] = prevalence

        auprc, low, high, prec_low, prec_high, mean_recall = calculate_auprc_ci(d['y_true'], d['y_pred'])
        precision, recall, _ = precision_recall_curve(d['y_true'], d['y_pred'])

        ax.plot(recall, precision,
                color=CLINICAL_COLORS['MIMIC'], lw=1,
                label=f'MIMIC (AUPRC: {auprc:.2f} [{low:.2f}-{high:.2f}])')
        ax.fill_between(mean_recall, prec_low, prec_high,
                        color=CLINICAL_COLORS['MIMIC'], alpha=0.2)

    # --- Prevalence baselines (equivalent of the diagonal in ROC) ---
    for name, prev in prevalences.items():
        ax.axhline(y=prev, color=CLINICAL_COLORS[name],
                   linestyle=':', alpha=0.4, lw=1,
                   label=f'{name} prevalence ({prev:.1%})')

    # --- Formatting ---
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.0])
    ax.set_aspect('equal', 'box')
    ax.set_xlabel('Recall (Sensitivity)')
    ax.set_ylabel('Precision (Positive Predictive Value)')
    ax.legend(loc="upper right", fontsize=9, frameon=True,
              edgecolor='black', fancybox=False)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"PRC saved to {save_path}")
    plt.show()


if __name__ == "__main__":
    plot_precision_recall()